# VAPAR (UZH) + Spyx Architectures

This notebook targets the VAPAR dataset/repo from UZH.

Status in Tonic:
- No VAPAR dataset class in `tonic.datasets` in this environment.

Research focus:
- visual-attention or control-surrogate prediction
- standard vs ternary vs log-polar encoders
- Optuna tuning over foveated geometry and channel budgets


In [ ]:
from pathlib import Path

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import optuna

import spyx.models as fm

DATA_ROOT = Path("../data/VAPAR")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT.resolve())


def tonic_has_vapar():
    import tonic.datasets as d
    return any("vapar" in n.lower() for n in dir(d))

print("tonic_has_vapar =", tonic_has_vapar())
print("expected local artifacts from VAPAR/OSF: drone.csv, frame_index.csv, split*.csv")


In [ ]:
def synthetic_vapar_batch(batch=4, T=12, H=90, W=160, C=3, seed=3):
    rng = np.random.default_rng(seed)
    x = rng.random(size=(T, batch, H, W, C), dtype=np.float32)
    y = rng.normal(size=(batch, 3)).astype(np.float32)
    return jnp.asarray(x), jnp.asarray(y)


def vapar_model_bank(input_hw=(90, 160), input_channels=3):
    conv_cfg = fm.ConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=24, channels2=48, output_dim=3)
    logpolar_cfg = fm.LogPolarFoveatedConvConfig(input_hw=input_hw, input_channels=input_channels, radial_bins=24, angular_bins=48, channels1=24, channels2=48, output_dim=3)
    return {
        "conv_lif": lambda x: fm.ConvLIFSNN(conv_cfg)(x),
        "logpolar_foveated": lambda x: fm.LogPolarFoveatedConvSNN(logpolar_cfg)(x),
        "ternary_conv": lambda x: fm.TernaryConvLIFSNN(conv_cfg)(x),
    }


x, y = synthetic_vapar_batch()
for name, fn in vapar_model_bank().items():
    net = hk.without_apply_rng(hk.transform(fn))
    params = net.init(jax.random.PRNGKey(0), x)
    pred, aux = net.apply(params, x)
    mse = jnp.mean((pred - y) ** 2)
    print(name, pred.shape, float(mse), np.asarray(aux["spike_rate"]))


## Optuna Sweep

This search compares standard, ternary, and log-polar Spyx visual encoders for VAPAR-style attention or control-surrogate targets.


In [ ]:
def vapar_objective(trial):
    arch = trial.suggest_categorical("arch", ["conv_lif", "ternary_conv", "logpolar_foveated"])
    channels1 = trial.suggest_categorical("channels1", [16, 24, 32])
    channels2 = trial.suggest_categorical("channels2", [32, 48, 64])
    beta = trial.suggest_float("beta", 0.8, 0.98)

    x_local, y_local = synthetic_vapar_batch(seed=trial.number)

    if arch == "logpolar_foveated":
        cfg = fm.LogPolarFoveatedConvConfig(input_hw=(90, 160), input_channels=3, radial_bins=trial.suggest_categorical("radial_bins", [16, 24, 32]), angular_bins=trial.suggest_categorical("angular_bins", [32, 48, 64]), channels1=channels1, channels2=channels2, output_dim=3, beta=beta)
        forward = lambda xb: fm.LogPolarFoveatedConvSNN(cfg)(xb)
    else:
        cfg = fm.ConvConfig(input_hw=(90, 160), input_channels=3, channels1=channels1, channels2=channels2, output_dim=3, beta=beta)
        forward = (lambda xb: fm.TernaryConvLIFSNN(cfg)(xb)) if arch == "ternary_conv" else (lambda xb: fm.ConvLIFSNN(cfg)(xb))

    net = hk.without_apply_rng(hk.transform(forward))
    params = net.init(jax.random.PRNGKey(0), x_local)
    pred, aux = net.apply(params, x_local)
    mse = jnp.mean((pred - y_local) ** 2)
    spike_penalty = 0.01 * jnp.mean(jnp.asarray(aux["spike_rate"]))
    return float(mse + spike_penalty)


study = optuna.create_study(direction="minimize", study_name="vapar_spyx_arch_search")
study.optimize(vapar_objective, n_trials=8)
study.best_trial.params, study.best_value


## Next Steps for Real VAPAR Experiments

1. Use `frame_index.csv` and split files to build reproducible partitions.
2. Add gaze/attention auxiliary targets where available.
3. Compare log-polar and standard conv models under matched compute budgets.
